# 4 · Side information, clusters and the stash

Interactions are not the only thing a `Dataset` can carry. This guide adds three
more kinds of data to MovieLens-100K and follows each one to the models and
metrics that read it:

- **item side information** (genres and decade of release): the file WarpRec
  expects, how it trims the catalogue, `keep_unseen_items`, and the two views
  the `Dataset` builds from it, a feature matrix and per-column indices;
- **user and item clusters** (users by gender, items by genre): the files, and
  how the cluster labels are renumbered for the fairness metrics;
- **the stash**: any structure of your own, handed to every model and every
  metric as a keyword argument.

It ends by running two side-information models next to their collaborative
counterparts, evaluated with metrics that read the features, the clusters and
the stash.

**Requirements:** `pip install warprec jupyter`. Runs in about a minute.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch
from loguru import logger

sys.path.insert(0, "..")
from guide_data import GENRES, movielens_100k

folder = movielens_100k()
RUNS = Path("../runs/04-side-information-and-clusters")
RUNS.mkdir(parents=True, exist_ok=True)

`movielens_100k()` downloads the data once and writes `ratings.tsv`, `items.tsv`
and `users.tsv`, exactly as guide 1 does step by step. The item file holds the
genres as one `|`-joined column:

In [2]:
items = pd.read_csv(folder / "items.tsv", sep="\t")
users = pd.read_csv(folder / "users.tsv", sep="\t")
items.head(3)

,item_id,title,release_date,genres
0,1,Toy Story (1995),01-Jan-1995,Animation|Children's|Comedy
1,2,GoldenEye (1995),01-Jan-1995,Action|Adventure|Thriller
2,3,Four Rooms (1995),01-Jan-1995,Thriller


## Item side information

WarpRec reads side information from one file with **the item ID in the first
column** (named as `reader.labels.item_id_label`, `item_id` by default) and a
feature in every other column. How a column is interpreted depends on its type:

- a **numeric** column is one feature whose value is kept as it is, so a
  one-hot layout is preserved exactly;
- a **text** column is categorical and is expanded into one indicator feature
  per distinct value.

The file below uses both: the 19 genres as 0/1 columns, and the decade of
release as text. One film has no release date, so its decade is left empty.

In [3]:
release_year = pd.to_datetime(items.release_date, format="%d-%b-%Y").dt.year
side = pd.DataFrame({"item_id": items.item_id})
for genre in GENRES:
    side[genre] = items.genres.str.split("|").apply(lambda gs, g=genre: int(g in gs))
side["decade"] = (release_year // 10 * 10).astype("Int64").astype(str) + "s"
side.loc[release_year.isna(), "decade"] = None
side.to_csv(RUNS / "items-side.tsv", sep="\t", index=False)
side.iloc[:3, [0, 1, 2, 3, 4, 5, 6, -1]]

,item_id,unknown,Action,Adventure,Animation,Children's,Comedy,decade
0,1,0,0,0,1,1,1,1990s
1,2,0,1,1,0,0,0,1990s
2,3,0,0,0,0,0,0,1990s


Why not hand WarpRec the `genres` column as it is? A text cell is **one**
categorical value, so `Animation|Children's|Comedy` would become a feature of
its own, unrelated to `Animation|Comedy`. Counting the distinct values shows
how many features that would make, against 19 genres:

In [4]:
items.genres.nunique()

216

The configuration names the file under `reader.side`. `sep` and `header` default
to a tab and `true`; a file without a header lists its columns in
`column_names`, item ID first. A `parquet` file is read with
`file_format: parquet`.

In [5]:
print(Path("configs/side.yml").read_text())

# Item side information: one row per item, the item ID first, every other
# column a feature. Paths are relative to the guide's folder.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  side:
    local_path: ../runs/04-side-information-and-clusters/items-side.tsv
    sep: "\t"
    header: true
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  VSM:
    similarity: cosine
    user_profile: binary
    item_profile: binary
evaluation:
  top_k: [10]
  metrics: [nDCG]



The log of `initialize_datasets` is worth reading here. After the split it
reports how many items were dropped for lacking side information, the size of
the feature matrix, and one line per column with the number of distinct values
it found.

In [6]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration

config = load_design_configuration("configs/side.yml")
main, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)

08-10-2026 12:22:37 - WarpRec - 📝 Reading design configuration file in: configs/side.yml


08-10-2026 12:22:37 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:22:37 - WarpRec - 📝 Starting test splitting process with temporal_holdout splitting strategy.


08-10-2026 12:22:37 - WarpRec - 📝 Test splitting completed in : 0.01s


08-10-2026 12:22:37 - WarpRec - ✅ Splitting process over in 0.01s.


...

08-10-2026 12:22:38 - WarpRec - 📊 Number of users: 943      Number of items: 1648      Transactions: 89561


08-10-2026 12:22:38 - WarpRec - 📝 ------------------------------------------------------------------------


08-10-2026 12:22:38 - WarpRec - 📝 --------------------------------Test set--------------------------------


08-10-2026 12:22:38 - WarpRec - 📊 Number of users: 943      Number of items: 1335      Transactions: 10401


08-10-2026 12:22:38 - WarpRec - 📝 ------------------------------------------------------------------------


In [7]:
logger.disable("warprec")

### Two views of the same file

Model families consume attributes differently, so the `Dataset` builds two
structures from the file.

**The feature matrix** is an item × feature matrix: numeric columns keep their
value, text columns become one indicator per distinct value. Content-based and
hybrid models (`VSM`, `AttributeItemKNN`, `AttributeUserKNN`, `CEASE`,
`AddEASE`) take similarities over it; the models get it from
`train_set.get_side_sparse()`. `get_feature_matrix()` returns it as a dense
tensor with one extra all-zero row for the padding item, which is the form the
diversity metrics read.

In [8]:
matrix = main.get_feature_matrix()
_, item_map = main.get_mappings()
print(
    "feature matrix:",
    tuple(matrix.shape),
    "| info()['n_features']:",
    main.info()["n_features"],
)
print("sparse, as the models read it:", main.train_set.get_side_sparse().shape)

feature matrix: (1649, 27) | info()['n_features']: 27
sparse, as the models read it: (1648, 27)


Its columns follow the file: the 19 genre columns in order, then one column
per decade, sorted. Toy Story's row has three genres and one decade:

In [9]:
decades = sorted(side.decade.dropna().unique())
feature_names = GENRES + decades
toy_story = matrix[item_map[1]]
[feature_names[i] for i in toy_story.nonzero().flatten().tolist()]

['Animation', "Children's", 'Comedy', '1990s']

**The per-column indices** are what context-aware models (`DeepFM`, `NFM`,
`WideAndDeep`, `xDeepFM`, …) embed: each column of the file stays one
field, and each of its values gets an index, starting at 1 because 0 is kept
for unknown values. `info()["feature_dims"]` gives each field's vocabulary
size, which is what these models size their embedding tables from. A 0/1 genre
column has two values plus the reserved index; the decade has eight decades,
the placeholder used for the missing date, and the reserved index.

In [10]:
dims = main.info()["feature_dims"]
print(len(dims), "fields:", {k: dims[k] for k in ["Action", "Comedy", "decade"]})

20 fields: {'Action': 3, 'Comedy': 3, 'decade': 10}


`get_features_lookup()` is that encoding as an item-indexed tensor, padding
row included; the models read the same table from
`train_set.get_side_tensor()`. Toy Story again: `2` where the genre flag is 1, `1` where it is 0,
and the index of `1990s` in the decade field.

In [11]:
lookup = main.get_features_lookup()
print(tuple(lookup.shape))
dict(zip(GENRES + ["decade"], lookup[item_map[1]].int().tolist()))

(1649, 20)


{'unknown': 1,
 'Action': 1,
 'Adventure': 1,
 'Animation': 2,
 "Children's": 2,
 'Comedy': 2,
 'Crime': 1,
 'Documentary': 1,
 'Drama': 1,
 'Fantasy': 1,
 'Film-Noir': 1,
 'Horror': 1,
 'Musical': 1,
 'Mystery': 1,
 'Romance': 1,
 'Sci-Fi': 1,
 'Thriller': 1,
 'War': 1,
 'Western': 1,
 'decade': 8}

### Which items stay

Side information also decides the catalogue. **An item without a row in the
side file is removed together with all its interactions**, so that every model
in a run, with or without attributes, is compared on the same items. With the
full file nothing was dropped (`Filtered out 0 item_id` in the log above). With
a file that misses items, say metadata exported before the last 82 films were
added, the interactions on those films go too:

In [12]:
import narwhals as nw

from warprec.data import Dataset
from warprec.data.reader import LocalReader
from warprec.data.schema import SplitSpec
from warprec.data.splitting import Splitter

data = LocalReader().read_tabular(local_path=str(folder / "ratings.tsv"))
train, _, test = Splitter().split_transaction(
    data, test=SplitSpec(strategy="temporal_holdout", ratio=0.1)
)
partial_side = LocalReader().read_tabular(local_path=str(RUNS / "items-side.tsv"))
partial_side = partial_side.filter(nw.col("item_id") <= 1600)


def summary(dataset: Dataset) -> dict:
    """Catalogue size and interaction counts of a dataset."""
    return {
        "items": dataset.get_dims()[1],
        "train rows": dataset.train_set.get_sparse().nnz,
        "test rows": dataset.eval_set.get_sparse().nnz,
    }


pd.DataFrame(
    {
        "no side information": summary(Dataset(train_data=train, eval_data=test)),
        "items <= 1600 only": summary(
            Dataset(train_data=train, eval_data=test, side_data=partial_side)
        ),
    }
).T

,items,train rows,test rows
no side information,1648,89561,10401
items <= 1600 only,1584,89448,10388


The opposite case is an item that **has attributes but no training
interaction**. The temporal split leaves 34 films whose only ratings fall in
the test period; by default they are dropped like any item missing from the
training split. `keep_unseen_items: true` keeps them in the catalogue as
all-zero columns of the interaction matrix, so a model that scores from
attributes can still rank them while a collaborative model scores them last.

In [13]:
print(Path("configs/side-keep-unseen.yml").read_text())

# The same side information, keeping the items that have attributes but no
# training interaction.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  side:
    local_path: ../runs/04-side-information-and-clusters/items-side.tsv
    keep_unseen_items: true
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  VSM:
    similarity: cosine
    user_profile: binary
    item_profile: binary
evaluation:
  top_k: [10]
  metrics: [nDCG]



In [14]:
config = load_design_configuration("configs/side-keep-unseen.yml")
kept, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
interactions = kept.train_set.get_sparse()
pd.DataFrame(
    {
        "default": {**summary(main), "items without training rows": 0},
        "keep_unseen_items": {
            **summary(kept),
            "items without training rows": int(
                (interactions.getnnz(axis=0) == 0).sum()
            ),
        },
    }
).T

,items,train rows,test rows,items without training rows
default,1648,89561,10401,0
keep_unseen_items,1682,89561,10401,34


The test rows are the same in both: a test interaction on an item the split
held out of training is still dropped, so these items can be recommended but
never count as hits. Evaluating on them is what the cold-start splitting
strategies are for (guide 13).

## User and item clusters

Clusters assign each user, or each item, to a group. The fairness and bias
metrics (guide 12) compare groups, so they read them. A cluster file has two
columns: the ID, named like the ID column of the interactions, and the
cluster, named `cluster` unless `reader.labels.cluster_label` says otherwise.
Here users are grouped by gender and items by the first genre they list.

In [15]:
user_clusters = pd.DataFrame({"user_id": users.user_id, "cluster": users.gender})
item_clusters = pd.DataFrame(
    {"item_id": items.item_id, "cluster": items.genres.str.split("|").str[0]}
)
user_clusters.to_csv(RUNS / "user-gender.tsv", sep="\t", index=False)
item_clusters.to_csv(RUNS / "item-first-genre.tsv", sep="\t", index=False)
user_clusters.head(3)

,user_id,cluster
0,1,M
1,2,F
2,3,M


Both files are configured under `reader.clustering`, each with its own path,
separator, header flag and format. The cluster column is read as `int32`
unless `reader.dtypes.cluster_type` says otherwise; these clusters are labels,
so they are read as `str`.

In [16]:
print(Path("configs/clusters.yml").read_text())

# User and item clusters: one file each, an ID column and a cluster column.
# The cluster column's name and type are shared by both files.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  labels:
    cluster_label: cluster
  dtypes:
    cluster_type: str
  clustering:
    user_local_path: ../runs/04-side-information-and-clusters/user-gender.tsv
    item_local_path: ../runs/04-side-information-and-clusters/item-first-genre.tsv
    user_sep: "\t"
    item_sep: "\t"
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



In [17]:
config = load_design_configuration("configs/clusters.yml")
clustered, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
list(clustered.user_cluster.items())[:3]

[(0, 'M'), (1, 'F'), (2, 'M')]

`user_cluster` and `item_cluster` keep your labels, keyed by user and item
index. What the metrics read is `get_user_cluster()` and `get_item_cluster()`:
one integer per user or item, with the distinct labels **renumbered from 1 in
sorted order**. `0` is the fallback for a user or item the file does not
mention, so a file that numbers its clusters from 0 is shifted up rather than
merged with the fallback. The item tensor carries one more entry, `0`, for the
padding item.

In [18]:
codes = clustered.get_user_cluster()
labels = [clustered.user_cluster[u] for u in range(len(codes))]
pd.crosstab(pd.Series(labels, name="label"), pd.Series(codes.tolist(), name="code"))

code,1,2
label,,
F,273,0
M,0,670


In [19]:
item_codes = clustered.get_item_cluster()
item_labels = [
    clustered.item_cluster.get(i, "(padding)") for i in range(len(item_codes))
]
counts = pd.DataFrame(
    {"label": item_labels, "code": item_codes.tolist()}
).value_counts()
print(tuple(item_codes.shape), "for", clustered.get_dims()[1], "items")
counts.sort_index(level="code").head(5)

(1649,) for 1648 items


label       code
(padding)   0         1
Action      1       248
Adventure   2        60
Animation   3        35
Children's  4        48
Name: count, dtype: int64

## The stash

Anything WarpRec has no section for (a tensor of item prices, a precomputed
similarity, a per-user weight) goes in the dataset's **stash**:
`add_to_stash(key, value)` stores it and `get_stash()` returns the dictionary.
The pipelines unpack that dictionary into the constructor of every model
(`model_registry.get(..., **dataset.get_stash())`) and pass it to the evaluator,
which unpacks it into the constructor of every metric. A model or metric of your
own picks its entry up as a keyword argument; the built-in ones ignore keys
they do not know.

Here the stash gets each film's release year. A structure meant for models is
indexed like everything else in the dataset, so it is built through the item
mapping, with an extra slot for the padding item.

In [20]:
year_by_id = dict(zip(items.item_id, release_year))
item_year = torch.full((main.get_dims()[1] + 1,), float("nan"))
for raw_id, index in item_map.items():
    item_year[index] = year_by_id[raw_id]

main.add_to_stash("item_year", item_year)
{key: tuple(value.shape) for key, value in main.get_stash().items()}

{'item_year': (1649,)}

Each `Dataset` has its own stash: the main dataset, the validation dataset and
every fold are separate objects, and a structure built from one training split
is wrong for another. That is why the usual place to fill it is the
`on_dataset_creation` callback, which receives all of them (guide 16). Keys are
unpacked next to the arguments WarpRec passes itself, so pick names no model or
metric parameter uses: a key `k` makes every metric fail to build, and a key
`num_items` silently replaces the catalogue size every metric is given.

A metric that reads `item_year` takes it as a constructor argument. This one
reports the mean release year of each user's top-k; guide 15 covers writing
metrics properly.

In [21]:
from warprec.evaluation.metrics.base_metric import UserAverageTopKMetric
from warprec.utils.enums import MetricBlock
from warprec.utils.registry import metric_registry


@metric_registry.register("MeanReleaseYear")
class MeanReleaseYear(UserAverageTopKMetric):
    """Mean release year of the items in the top-k, read from the stash."""

    _REQUIRED_COMPONENTS = {
        MetricBlock.BINARY_RELEVANCE,
        MetricBlock.VALID_USERS,
        MetricBlock.TOP_K_INDICES,
        MetricBlock.TOP_K_BINARY_RELEVANCE,
    }

    def __init__(self, k, num_users, item_year, **kwargs):
        super().__init__(k=k, num_users=num_users)
        self.register_buffer("item_year", item_year)

    def compute_scores(self, preds, target, top_k_rel, **kwargs):
        top_k = kwargs[f"top_{self.k}_indices"]
        return self.item_year[top_k].nanmean(dim=1)

## Side information reaching a model

The last configuration reads the side information and both cluster files, and
names two pairs of closed-form models. In each pair the second model is the
first one with the attributes added:

- `ItemKNN` takes item–item similarities over the interactions,
  `AttributeItemKNN` over the feature matrix;
- `EASE` learns an item–item weight matrix from the interactions, `CEASE`
  appends the item features to the interactions, weighted by `alpha`, before
  solving the same problem.

The metrics read the other structures: `SRecall` the feature matrix,
`UserMADRanking` the user clusters.

In [22]:
print(Path("configs/models.yml").read_text())

# Side information and clusters together, and two pairs of models: each
# side-information model next to its collaborative counterpart.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  dtypes:
    cluster_type: str
  side:
    local_path: ../runs/04-side-information-and-clusters/items-side.tsv
  clustering:
    user_local_path: ../runs/04-side-information-and-clusters/user-gender.tsv
    item_local_path: ../runs/04-side-information-and-clusters/item-first-genre.tsv
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  ItemKNN:
    k: 50
    similarity: cosine
  AttributeItemKNN:
    k: 50
    similarity: cosine
  EASE:
    l2: 200.0
  CEASE:
    l2: 200.0
    alpha: 1.0
evaluation:
  top_k: [10]
  metrics: [nDCG, SRecall, UserMADRanking]



Each model is built from the dataset as the pipelines build it: the training
interactions, `info()`, and the stash. The evaluator receives what the
pipelines' `build_evaluator` passes it: the feature matrix, both cluster
tensors and the stash.

In [23]:
from warprec.evaluation.evaluator import Evaluator
from warprec.utils.registry import model_registry

config = load_design_configuration("configs/models.yml")
dataset, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
dataset.add_to_stash("item_year", item_year)

results = {}
for name, params in config.models.items():
    model = model_registry.get(
        name,
        params=params,
        info=dataset.info(),
        interactions=dataset.train_set,
        seed=42,
        **dataset.get_stash(),
    )
    evaluator = Evaluator(
        config.evaluation.metrics + ["MeanReleaseYear"],
        config.evaluation.top_k,
        train_set=dataset.train_set.get_sparse(),
        additional_data=dataset.get_stash(),
        feature_lookup=dataset.get_feature_matrix(),
        user_cluster=dataset.get_user_cluster(),
        item_cluster=dataset.get_item_cluster(),
    )
    evaluator.evaluate(model, dataset.get_evaluation_dataloader(), "full", dataset)
    scores = evaluator.compute_results()[10]
    # Per-user metrics come back as one value per user, global ones as a number.
    results[name] = {
        metric: value.nanmean().item() if torch.is_tensor(value) else value
        for metric, value in scores.items()
    }

pd.DataFrame(results).T.round(4)

,nDCG,SRecall,UserMADRanking,MeanReleaseYear
ItemKNN,0.1331,0.2591,0.0043,1989.0509
AttributeItemKNN,0.0157,0.0357,0.0058,1994.1947
EASE,0.1542,0.2843,0.0058,1989.8695
CEASE,0.1553,0.2842,0.0047,1989.7992


The attributes clearly reach the models. `AttributeItemKNN`, which replaces
interaction similarity with genre-and-decade similarity, ranks far worse on
nDCG and SRecall than `ItemKNN`, and recommends noticeably newer films (a
higher mean release year). `CEASE` changes `EASE` only slightly at this
`alpha`; how much weight the features deserve is a hyperparameter to search on
validation data (guide 10), not on the test set. Guide 12 explains the
metrics; `UserMADRanking` is, with two groups, the absolute difference between the
mean nDCG of female and of male users.

## Summary

| key | what it does |
|---|---|
| `reader.side.local_path`, `sep`, `header`, `file_format` | where and how to read the item side information; item ID first, one feature per other column |
| `reader.side.column_names` | the column names of a file without a header, item ID first |
| `reader.side.keep_unseen_items` | keep items that have attributes but no training interaction, as all-zero columns |
| `reader.clustering.user_local_path`, `item_local_path` | the user and the item cluster files |
| `reader.clustering.user_sep`, `item_sep`, `user_header`, `item_header`, `user_file_format`, `item_file_format` | how each cluster file is read |
| `reader.labels.cluster_label` | the name of the cluster column in both files (default `cluster`) |
| `reader.dtypes.cluster_type` | the type the cluster column is read as (default `int32`) |

The file formats are described in [Readers](https://warprec.readthedocs.io/en/latest/data-management/reader/),
the keys in [Reader configuration](https://warprec.readthedocs.io/en/latest/configuration/reader/)
and the stash in [Stash](https://warprec.readthedocs.io/en/latest/data-management/stash/).
The models that read side information are listed under
[Recommenders](https://warprec.readthedocs.io/en/latest/recommenders/).